# AXIOMchat — Multi-Facet Topic-Drift Detection
### A research walkthrough, not a shipped tool

This project started as a DAG-based context-*pruning* prototype, then a
single-fixed-anchor drift detector. Both were replaced after real testing
exposed real flaws — see `FINDINGS.md` for the full account, including
negative results. The current design: extract a **set** of distinct session
facets once (not one blended point), score every message by its best match
across all facets through a cheap-to-expensive cascade, and **never delete
anything** — every path returns a score to the caller, who decides what to
do with it.

This notebook runs that pipeline end-to-end with mocked embeddings/LLM calls
(no Ollama server or API key required, fully reproducible). The real
findings — what happened when this was tested against an actual 170-message
conversation with a real embedder and a real local LLM — are documented in a
markdown section near the end, since those runs took minutes each and aren't
practical to re-run inline here.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import pandas as pd
import matplotlib.pyplot as plt

from src.anchor import AnchorExtractor
from src.datahub import Datahub
from src.embedder import MockEmbedder
from src.llm_client import MockLLMClient
from src.relevance import DriftScorer

embedder = MockEmbedder()
datahub = Datahub("../data/datahub.json")
llm_client = MockLLMClient()

print("Setup complete.")

## 2. Multi-facet anchor extraction

`AnchorExtractor` waits for `k` user messages, strips structured-inventory
lines (e.g. a resume's skills table — see `FINDINGS.md` #13 for why), then
runs a two-stage pipeline: `summarize()` (detail-preserving) followed by
`extract_facets()` — a short **list** of distinct topics/tasks, not one
blended sentence. A single averaged anchor couldn't represent a session that
covers several real facets (see `FINDINGS.md` #11); this returns several,
and a message is later scored against whichever one it's actually about.

In [ ]:
with open("../examples/sample_conversation.json") as f:
    messages = json.load(f)

anchor_texts = AnchorExtractor(llm_client, datahub, k=2).extract(messages)
print(f"Anchors ({len(anchor_texts)}):")
for a in anchor_texts:
    print(" -", a[:100])

## 3. Scoring cascade

`DriftScorer.score_message()` runs three stages, cheapest first:
1. `Datahub.is_noise` — filters greetings/acks before anything else runs
2. Cosine similarity against every anchor facet, keeping the **best** match
   — bands into `unrelated` / `ambiguous` / `relevant`
3. LLM disambiguation — **only** for the `ambiguous` band, considering all
   facets, bounding how many LLM calls a session ever needs

In [ ]:
scorer = DriftScorer(embedder, datahub, llm_client, anchor_texts)

rows = []
for m in messages:
    result = scorer.score_message(m["text"])
    rows.append({
        "id": m["id"],
        "role": m["role"],
        "text": m["text"][:50],
        "band": result.band,
        "cosine_score": result.cosine_score,
        "matched_facet": (result.matched_facet or "")[:40],
        "llm_verdict": result.llm_verdict,
    })

df = pd.DataFrame(rows)
df

In [ ]:
colors = {"unrelated": "#E05C5C", "ambiguous": "#E8A33D", "relevant": "#4C9BE8"}
scored = df.dropna(subset=["cosine_score"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(scored["id"], scored["cosine_score"], color=[colors[b] for b in scored["band"]])
ax.set_title("Best cosine similarity across all facets, by message (mock embedder)")
ax.set_ylabel("cosine similarity")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Band counts:", df["band"].value_counts().to_dict())

## 4. Real findings (not reproducible offline)

The mocked run above proves the *mechanism* works. It doesn't prove
real-world accuracy — that was tested separately against a real ~170-message
conversation, a real embedder, and a real local LLM (`llama3.1:8b` via
Ollama). Full detail, including the negative results, is in `FINDINGS.md`;
the headline arc:

- **A single fixed anchor structurally can't represent a multi-facet
  session.** Even with a good, distilled anchor and the real embedder,
  86% of real messages landed `unrelated`, including unambiguously on-topic
  ones (*"ok, so I can keep the parallel computing right"* scored 0.220).
- **Splitting the anchor into multiple facets, scored by best match,
  measurably fixed it.** That exact miss now scores 0.618 and correctly
  lands `relevant`, matched against a real, LLM-identified "parallel
  computing" facet. `unrelated` dropped from 86% to 67.6%.
- **Naively asking for more/fewer facets made things worse, not better.**
  Capping the count lost specificity (real projects became vague
  categories); asking for a wider range caused the model to enumerate
  individual tool names straight out of a resume's skills table ("VS
  Code", "Google Colab") as if they were discussion topics — a local 8B
  model doesn't reliably hold multiple simultaneous prompt constraints.
- **The actual fix was cleaning the input, not tuning the prompt further**:
  stripping structured-inventory lines (like a skills table) before
  summarization, so individual tools stop being mistaken for topics. That
  produced a clean 12-facet list with zero tool-name pollution.
- **Real, unresolved cost**: multi-facet scoring escalates more messages to
  the LLM disambiguation stage (more accuracy, but each call also got
  slower, since the prompt now lists every facet) — several minutes per
  full run on a local 8B model.

## Open questions

See `README.md` ("Open items") and `FINDINGS.md` ("What's still
unverified") for the current honest state — including whether new facets
should be allowed to accumulate as a long session evolves, `AnthropicClient`
never having been run, and FTS5 sitting unused.